# Sesión 2 — Sesgo-Varianza, Validación, Árboles y Ensambles

Código de los conceptos de la sesión y el ejercicio para practicarlos.

## Retomamos: dataset y modelos de la Sesión 1

Esta sesión continúa directamente sobre la clasificación que vimos en la
Sesión 1: **German Credit Data** — un banco quiere predecir el riesgo de
que un solicitante de crédito no pague (`logreg` y `rf`, clase objetivo
`bad`/`good`). Recargamos rápidamente los mismos datos y modelos para
que este notebook corra de forma independiente.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

COLOR = {
    "blue": "#2a78d6",
    "orange": "#eb6834",
    "aqua": "#1baf7a",
    "gray": "#898781",
    "grid": "#e1e0d9",
}
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = COLOR["grid"]
plt.rcParams["axes.edgecolor"] = COLOR["gray"]

# Mismos datos y modelos de la Sesión 1 (German Credit Data)
d = fetch_openml("credit-g", version=1, as_frame=True, parser="auto")
df = d.frame
y = (df["class"] == "bad").astype(int)
X = pd.get_dummies(df.drop(columns=["class"]), drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print("Retomamos German Credit Data:", X.shape, "· modelos logreg y rf re-entrenados.")

## 1. Sesgo vs. Varianza

Todo modelo comete error por dos razones distintas (más el ruido irreducible
de los datos):

- **Sesgo (bias)**: el error que viene de las **suposiciones simplificadoras**
  del modelo. Un modelo con alto sesgo tiende a **ignorar información** real
  presente en los datos y a aprender relaciones incorrectas o demasiado
  simples (p. ej., asumir una frontera de decisión lineal cuando la real es
  curva). Es un problema estructural del modelo, no de los datos de
  entrenamiento en particular.
- **Varianza**: qué tanto **cambiaría el modelo ajustado** si lo
  entrenáramos con una muestra de entrenamiento distinta. Un modelo con alta
  varianza es muy **sensible a pequeños cambios** en los datos de
  entrenamiento — puede ajustar el ruido específico de esa muestra en vez del
  patrón general.

**Regla general**: entre más flexible (compleja) es la familia de modelos,
menor es su sesgo (puede representar relaciones más ricas) pero mayor es su
varianza (tiene más "grados de libertad" para sobreajustarse a la muestra
particular de entrenamiento). El error total esperado sobre datos nuevos se
puede descomponer, de forma simplificada, como:

$$\text{Error esperado} \approx \text{Sesgo}^2 + \text{Varianza} + \text{Error irreducible}$$

Esto genera la clásica curva en forma de "U" del error de prueba en función
de la complejidad del modelo, con un punto óptimo en el medio:

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
complejidad = np.linspace(0, 10, 200)
sesgo2 = 1.8 * ((10 - complejidad) / 10) ** 2
varianza = 1.8 * (complejidad / 10) ** 2
error_total = sesgo2 + varianza + 0.3

ax.plot(complejidad, sesgo2, color=COLOR["blue"], linewidth=2, label="Sesgo$^2$")
ax.plot(complejidad, varianza, color=COLOR["orange"], linewidth=2, label="Varianza")
ax.plot(complejidad, error_total, color="#0b0b0b", linewidth=2.5, label="Error total esperado")

optimo = complejidad[np.argmin(error_total)]
ax.axvline(optimo, linestyle="--", color=COLOR["gray"])
ax.text(optimo + 0.15, ax.get_ylim()[1] * 0.9, "óptimo", color=COLOR["gray"])

ax.set_xlabel("Complejidad / flexibilidad del modelo")
ax.set_ylabel("Error esperado")
ax.set_title("Trade-off sesgo-varianza (gráfico conceptual)")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.show()

En nuestro caso de estudio, la **regresión logística** (modelo lineal)
está más hacia la izquierda de este eje: menos flexible, más sesgo, menos
varianza. El **Random Forest** sin restricciones está más hacia la derecha:
muy flexible, menos sesgo, pero más riesgo de alta varianza / overfitting.

## 2. Overfitting y Underfitting

- **Overfitting (sobreajuste)**: el modelo se ajusta muy bien a los datos de
  entrenamiento (error de entrenamiento bajo) pero **no generaliza** a datos
  nuevos (error de prueba alto). Causas típicas: modelo demasiado complejo
  para la cantidad de datos disponible, datos ruidosos, muy pocos datos de
  entrenamiento. Soluciones típicas: simplificar el modelo, conseguir más
  datos, reducir el ruido/las variables irrelevantes, o aplicar
  **regularización**.
- **Underfitting (subajuste)**: el modelo ni siquiera se ajusta bien a los
  datos de entrenamiento (error de entrenamiento ya alto). Causas típicas:
  modelo demasiado simple, features poco informativas. Soluciones típicas:
  usar un modelo más flexible, construir mejores features, relajar
  restricciones de regularización.

Podemos ilustrar esto directamente sobre nuestros datos, variando la
**profundidad máxima** de un árbol de decisión (un solo parámetro que
controla su complejidad) y observando el error de entrenamiento vs. el error
de prueba:

In [ ]:
profundidades = range(1, 21)
error_train, error_test = [], []

for prof in profundidades:
    arbol = DecisionTreeClassifier(max_depth=prof, random_state=RANDOM_STATE)
    arbol.fit(X_train, y_train)
    error_train.append(1 - accuracy_score(y_train, arbol.predict(X_train)))
    error_test.append(1 - accuracy_score(y_test, arbol.predict(X_test)))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(list(profundidades), error_train, marker="o", color=COLOR["blue"], label="Error de entrenamiento")
ax.plot(list(profundidades), error_test, marker="o", color=COLOR["orange"], label="Error de prueba (test)")

mejor_prof = list(profundidades)[int(np.argmin(error_test))]
ax.axvline(mejor_prof, linestyle="--", color=COLOR["gray"])
ax.text(mejor_prof + 0.2, max(error_test) * 0.95, f"mínimo test\n(depth={mejor_prof})", color=COLOR["gray"])

ax.set_xlabel("max_depth (complejidad del árbol)")
ax.set_ylabel("Error de clasificación (1 - accuracy)")
ax.set_title("Overfitting / Underfitting: árbol de decisión sobre German Credit")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Se observa el patrón típico: con `max_depth` muy bajo, tanto el error de
entrenamiento como el de prueba son altos (**underfitting** — el árbol es
demasiado simple para capturar el patrón). A medida que aumentamos la
profundidad, el error de entrenamiento sigue bajando (incluso hasta casi 0),
pero el error de prueba deja de mejorar y eventualmente empieza a subir
(**overfitting** — el árbol empieza a memorizar particularidades de la
muestra de entrenamiento que no generalizan).

## 3. Interpretabilidad vs. Precisión

Existe con frecuencia un **trade-off** entre qué tan fácil es *explicar* un
modelo y qué tan *preciso* puede llegar a ser:

- Modelos **simples e interpretables** (regresión lineal/logística, árboles
  poco profundos): es fácil explicar exactamente cómo cada variable afecta la
  predicción (un coeficiente, una regla). Muy valorado en contextos regulados
  (riesgo crediticio, salud, sector público), donde hay que **justificar**
  una decisión ante un cliente o un regulador. A cambio, suelen tener mayor
  sesgo (menor precisión potencial).
- Modelos **complejos y de caja negra** (random forest, gradient boosting,
  redes neuronales): suelen lograr mayor precisión al capturar relaciones no
  lineales e interacciones entre variables, pero es mucho más difícil
  explicar *por qué* el modelo tomó una decisión particular para un caso
  específico.

Podemos ilustrar esto comparando los **coeficientes** de la regresión
logística (fácilmente interpretables) contra la **importancia de variables**
del Random Forest:

In [ ]:
coefs = pd.Series(logreg.named_steps["clf"].coef_[0], index=X.columns).sort_values()
top_coefs = pd.concat([coefs.head(5), coefs.tail(5)])

importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors_coef = [COLOR["blue"] if v < 0 else COLOR["orange"] for v in top_coefs.values]
axes[0].barh(top_coefs.index, top_coefs.values, color=colors_coef)
axes[0].set_title("Regresión logística: coeficientes\n(interpretable — dirección y magnitud claras)")
axes[0].axvline(0, color="#0b0b0b", linewidth=0.8)
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].barh(importancias.index[::-1], importancias.values[::-1], color=COLOR["orange"])
axes[1].set_title("Random Forest: importancia de variables\n(no indica dirección del efecto)")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

En la regresión logística podemos decir, por ejemplo, "un
`checking_status` con más fondos disponibles **reduce** la probabilidad de
`bad` en tal magnitud" — una explicación directa y auditable. En el Random
Forest solo sabemos **qué tan importante** fue cada variable para reducir la
impureza de los árboles, pero no la dirección ni la magnitud exacta del
efecto — de ahí que se necesiten técnicas adicionales (SHAP, LIME, etc., que
verás más adelante en el curso) para interpretar modelos de caja negra.

## 4. Validación cruzada

### ¿Por qué no basta con el error de entrenamiento?

El error medido sobre los mismos datos con los que se entrenó el modelo
(**error de entrenamiento**) **subestima sistemáticamente** el error real
sobre datos nuevos, porque el modelo tuvo la oportunidad de "memorizar"
particularidades de esa muestra. Necesitamos estimar el error sobre datos que
el modelo **no vio** durante el entrenamiento — eso es exactamente lo que
hace un conjunto de prueba (test), pero un solo split train/test depende de
*qué* observaciones cayeron en cada partición (tiene su propia varianza). La
**validación cruzada** promedia esta estimación sobre varias particiones
distintas, dando una medida más estable y confiable.

### Leave-One-Out Cross-Validation (LOOCV)

Con $n$ observaciones, se entrena el modelo $n$ veces: en cada iteración se
deja **una sola observación** fuera como prueba, y se entrena con las $n-1$
restantes. El error final es el promedio de los $n$ errores individuales.

- **Ventaja**: usa casi todos los datos para entrenar en cada iteración →
  estimación del error con **muy poco sesgo**.
- **Desventaja**: computacionalmente **muy costoso** (hay que entrenar $n$
  modelos) y, además, las $n$ estimaciones están muy correlacionadas entre sí
  (los conjuntos de entrenamiento se solapan casi por completo), por lo que
  la estimación final puede tener **alta varianza**.

### K-Fold Cross-Validation

Se divide aleatoriamente el dataset en $k$ particiones ("folds") de tamaño
aproximadamente igual. En cada una de las $k$ iteraciones, se usa un fold
distinto como prueba y los $k-1$ restantes como entrenamiento. El resultado
final es el promedio (y la desviación estándar, que nos da una idea de la
variabilidad) de las $k$ métricas obtenidas.

- Es un **balance costo/varianza**: computacionalmente mucho más barato que
  LOOCV (solo $k$ modelos, no $n$), y con menor varianza en la estimación
  final que LOOCV, a cambio de un poco más de sesgo.
- LOOCV es, de hecho, el caso particular $k = n$.
- En la práctica se suele usar $k = 5$ o $k = 10$: es el balance que mejor
  funciona empíricamente entre sesgo y varianza de la estimación, con un
  costo computacional razonable.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores_kfold = cross_val_score(logreg, X, y, cv=kf, scoring="f1")

print("F1 por fold (K=5):", np.round(scores_kfold, 3))
print(f"F1 promedio: {scores_kfold.mean():.3f}  (desviación estándar: {scores_kfold.std():.3f})")

f1_single_split = f1_score(y_test, logreg.predict(X_test))
print(f"\nF1 de una sola partición train/test: {f1_single_split:.3f}")

Nota cómo el F1 de una sola partición train/test puede diferir bastante
del promedio de K-Fold — y ese promedio viene acompañado de una desviación
estándar que nos dice **qué tan estable** es esa estimación. Ahora veamos
LOOCV sobre el mismo modelo (con `n_jobs=-1` para paralelizar, ya que implica
entrenar 1000 modelos):

In [ ]:
loo_scores = cross_val_score(logreg, X, y, cv=LeaveOneOut(), scoring="accuracy", n_jobs=-1)
print(f"LOOCV (n={len(loo_scores)} folds) — accuracy promedio: {loo_scores.mean():.3f}")

kfold_acc = cross_val_score(logreg, X, y, cv=kf, scoring="accuracy")
print(f"K-Fold (k=5) — accuracy promedio: {kfold_acc.mean():.3f}  (std: {kfold_acc.std():.3f})")

En este caso, LOOCV y K-Fold dan estimaciones muy similares del error
promedio (poco sesgo en ambos), pero K-Fold es órdenes de magnitud más barato
computacionalmente y además nos entrega directamente una desviación estándar
entre folds (con LOOCV, con `n=1000` folds de tamaño 1, esa desviación es
mucho menos informativa fold a fold). Por eso K-Fold con k=5 o 10 es, en la
práctica, la opción por defecto casi siempre.

---

Con esto cerramos la evaluación de un solo modelo con distintas técnicas.
Ahora construimos, desde la base, el modelo que motivó buena parte de la
discusión de hoy: árboles de decisión, y cómo combinarlos en ensambles.
Pasamos a un dataset nuevo (detección de fraude con tarjetas de crédito).

## Preparación: librerías

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.tree import DecisionTreeClassifier, plot_tree

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (7, 5)

## 5. Árboles de decisión

### 1.1 Idea general

Un árbol de decisión **particiona el espacio de entrada** (el espacio de las
variables predictoras) en regiones rectangulares disjuntas, mediante una
secuencia de preguntas binarias del tipo `¿variable_j <= umbral?`. Cada región
resultante (hoja del árbol) recibe una **predicción constante**:

- En **regresión**: el promedio de `y_fraude` de las observaciones que caen en esa
  hoja.
- En **clasificación**: la clase mayoritaria (o la distribución de
  probabilidades) de las observaciones que caen en esa hoja.

El árbol se construye de forma **greedy** (voraz): en cada nodo se busca la
variable y el umbral que producen la mejor mejora en una **función de costo**,
sin garantizar que la secuencia completa de splits sea globalmente óptima.

### 1.2 Función de costo

**Regresión — varianza / MSE dentro del nodo.** Para un nodo con
observaciones $\{y_i\}$, el costo es la varianza (equivalente a MSE respecto
a la media del nodo):

$$
\text{MSE}(\text{nodo}) = \frac{1}{N} \sum_{i=1}^{N} (y_i - \bar{y})^2
$$

Se busca el split que minimiza el promedio ponderado del MSE de los dos nodos
hijos.

**Clasificación — Índice de Gini.** Sea $p_j$ la proporción de observaciones
de la clase $j$ en el nodo:

$$
\text{Gini} = 1 - \sum_{j} p_j^2
$$

- Gini = 0 → nodo perfectamente puro (una sola clase).
- Gini más alto → más mezcla de clases.
- **Menor Gini = mayor pureza.** El árbol busca el split que más reduce el
  Gini promedio (ponderado por tamaño) de los nodos hijos respecto al nodo
  padre.

**Clasificación — Entropía y ganancia de información.**

$$
\text{Entropía} = - \sum_{j} p_j \log_2(p_j)
$$

- Entropía = 0 → nodo puro.
- **Mayor entropía = mayor desorden = menor ganancia de información** al usar
  ese split.
- La **ganancia de información** de un split es la reducción de entropía del
  padre respecto al promedio ponderado de entropía de los hijos.

Gini y Entropía suelen dar árboles muy similares en la práctica; Gini es
ligeramente más rápida de calcular (no requiere logaritmos) y es el default
en scikit-learn.

### 1.3 Ejemplo pequeño: ¿jugamos fútbol según el clima?

Para ilustrar cómo se elige la variable/split en un nodo, construimos a mano
un dataset diminuto (versión simplificada del clásico ejemplo de libros de
texto de árboles de decisión).

In [ ]:
futbol = pd.DataFrame({
    "clima": ["soleado", "soleado", "nublado", "lluvia", "lluvia", "lluvia",
              "nublado", "soleado", "soleado", "lluvia", "soleado", "nublado",
              "nublado", "lluvia"],
    "viento": ["debil", "fuerte", "debil", "debil", "debil", "fuerte",
               "fuerte", "debil", "debil", "debil", "fuerte", "fuerte",
               "debil", "fuerte"],
    "juega": ["no", "no", "si", "si", "si", "no", "si", "no", "si", "si",
              "si", "si", "si", "no"],
})
futbol

Calculemos el Gini del nodo raíz (sin ningún split) y comparémoslo con el
Gini ponderado que resulta de partir por `clima` vs. por `viento`. La
variable que produzca **mayor reducción de Gini** es la que el árbol elegiría
para el primer split.

In [ ]:
def gini(serie):
    p = serie.value_counts(normalize=True)
    return 1 - np.sum(p ** 2)

def gini_ponderado(df_fraude, variable, objetivo="juega"):
    total = len(df_fraude)
    costo = 0.0
    for valor, grupo in df_fraude.groupby(variable):
        costo += (len(grupo) / total) * gini(grupo[objetivo])
    return costo

gini_raiz = gini(futbol["juega"])
gini_clima = gini_ponderado(futbol, "clima")
gini_viento = gini_ponderado(futbol, "viento")

print(f"Gini del nodo raíz (sin split):      {gini_raiz:.3f}")
print(f"Gini ponderado al partir por clima:   {gini_clima:.3f}  -> reducción = {gini_raiz - gini_clima:.3f}")
print(f"Gini ponderado al partir por viento:  {gini_viento:.3f}  -> reducción = {gini_raiz - gini_viento:.3f}")

`clima` produce mayor reducción de Gini que `viento`, así que el árbol
elegiría `clima` como primer split. Esto es exactamente lo que hace
`DecisionTreeClassifier(criterion="gini")` internamente, pero evaluando
**todas** las variables y **todos** los posibles umbrales en variables
continuas, en cada nodo, de forma recursiva.

### 1.4 Ventajas de los árboles

- **Interpretables**: se pueden visualizar y explicar a alguien no técnico
  (ej. "si la variable X es mayor que Y, entonces...").
- **No requieren normalización/escalado** de variables: las decisiones se
  basan en umbrales, no en distancias.
- **Manejan variables continuas y categóricas** de forma natural, y no
  asumen ninguna forma funcional (lineal, etc.) de la relación entre `X_fraude` y
  `y_fraude`.

### 1.5 Regularización: pre-pruning vs. post-pruning

Un árbol sin restricciones puede crecer hasta tener una hoja por cada
observación de entrenamiento → **overfitting** total (varianza altísima).
Dos estrategias para controlarlo:

- **Pre-pruning** (poda anticipada): limitar la complejidad *mientras* se
  construye el árbol, mediante hiperparámetros como `max_depth`,
  `min_samples_split`, `min_samples_leaf`, `max_leaf_nodes`.
- **Post-pruning** (poda posterior): dejar crecer el árbol completo y luego
  **podar** ramas que no aportan suficiente reducción de costo en un set de
  validación (ej. *cost-complexity pruning*, disponible en scikit-learn como
  `ccp_alpha`).

En la práctica, con datasets grandes, el pre-pruning (controlar
`max_depth` y compañía) es la estrategia más usada por su simplicidad y
menor costo computacional.

---

### 1.6 Cargando el dataset real: Credit Card Fraud Detection

Una entidad financiera / procesador de pagos necesita identificar
transacciones de tarjeta de crédito **fraudulentas en tiempo real**, para
bloquearlas antes de que se completen. Es un problema de clasificación
con un desbalance extremo: la inmensa mayoría de las transacciones son
legítimas.

In [ ]:
# Este fetch descarga (o usa caché local) el dataset completo de OpenML.
# Puede tardar uno o varios minutos la primera vez.
d_fraude = fetch_openml("creditcard", version=1, as_frame=True, parser="auto")
df_full = d_fraude.frame

print("Shape del dataset ORIGINAL (completo):", df_full.shape)
print(df_full["Class"].value_counts())

El dataset completo tiene **284,807 filas** y las columnas `V1`...`V28` (28
componentes obtenidas por PCA sobre las variables originales, anonimizadas
por confidencialidad), `Amount` (monto de la transacción) y `Class` (la
variable objetivo: `1` = fraude, `0` = transacción normal). Solo **492**
transacciones (0.17%) son fraude: un desbalance extremo.

### Submuestreo para la clase

Nos quedamos con:

- **Todos** los casos de fraude (`Class == 1`): 492 filas.
- Una muestra aleatoria (`random_state=42`) de **20,000** transacciones
  normales (`Class == 0`).

El resultado es un dataset de ~20,500 filas que sigue siendo fuertemente
desbalanceado (~2.4% positivos) — suficiente para ilustrar todas las
técnicas de esta sesión sin descargar/comitear 150 MB de datos.

In [ ]:
df_full["Class"] = df_full["Class"].astype(int)

fraude = df_full[df_full["Class"] == 1]
no_fraude = df_full[df_full["Class"] == 0].sample(n=20_000, random_state=RANDOM_STATE)

df_fraude = pd.concat([fraude, no_fraude], axis=0).sample(frac=1.0, random_state=RANDOM_STATE)
df_fraude = df_fraude.reset_index(drop=True)

print("Shape del dataset SUBMUESTREADO para la clase:", df_fraude.shape)
print(df_fraude["Class"].value_counts())
print(f"Porcentaje de fraude: {100 * df_fraude['Class'].mean():.2f}%")
df_fraude.head()

In [ ]:
X_fraude = df_fraude.drop(columns=["Class"])
y_fraude = df_fraude["Class"]

X_train_fraude, X_test_fraude, y_train_fraude, y_test_fraude = train_test_split(
    X_fraude, y_fraude, test_size=0.2, stratify=y_fraude, random_state=RANDOM_STATE
)

print("Train:", X_train_fraude.shape, " | Positivos en train:", y_train_fraude.sum())
print("Test: ", X_test_fraude.shape, " | Positivos en test:", y_test_fraude.sum())

### 1.7 Entrenando un árbol de decisión sobre el dataset de fraude

Entrenamos un `DecisionTreeClassifier` limitando la profundidad
(pre-pruning) para que además se pueda **visualizar**.

In [ ]:
arbol = DecisionTreeClassifier(
    criterion="gini",
    max_depth=4,
    min_samples_leaf=20,
    random_state=RANDOM_STATE,
)
arbol.fit(X_train_fraude, y_train_fraude)

y_pred_arbol = arbol.predict(X_test_fraude)

print(classification_report(y_test_fraude, y_pred_arbol, digits=3, target_names=["normal", "fraude"]))

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    arbol,
    feature_names=X_fraude.columns,
    class_names=["normal", "fraude"],
    filled=True,
    rounded=True,
    max_depth=3,          # limitamos lo que se dibuja para que sea legible
    fontsize=8,
    ax=ax,
)
ax.set_title("Árbol de decisión (max_depth=4) — dataset de fraude submuestreado")
plt.show()

In [ ]:
importancias = pd.Series(arbol.feature_importances_, index=X_fraude.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
importancias.head(10).sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.set_title("Feature importances — árbol de decisión")
ax.set_xlabel("Importancia (reducción de Gini acumulada)")
plt.tight_layout()
plt.show()

importancias.head(10)

Nota: `feature_importances_` en scikit-learn mide, para cada variable, la
reducción total de impureza (Gini) que aporta en todos los nodos donde se
usó, ponderada por la proporción de muestras que pasan por ese nodo. Es una
medida útil pero **no es causal** (no implica que la variable "cause" el
fraude).

---

## 6. Métodos de ensamble

Un árbol individual es fácil de interpretar pero **inestable**: pequeños
cambios en los datos de entrenamiento pueden producir árboles muy distintos
(alta varianza). Los métodos de ensamble combinan **muchos** modelos
"débiles" (usualmente árboles) para obtener un modelo agregado más robusto.
El objetivo general de un ensamble es reducir **varianza**, **sesgo**, o
ambos, dependiendo de la estrategia.

### 2.1 Bagging (Bootstrap Aggregating)

1. Se generan $B$ muestras **bootstrap** (muestreo con reemplazo, mismo
   tamaño que el dataset original) a partir del set de entrenamiento.
2. Se entrena un modelo (típicamente un árbol) en cada muestra bootstrap,
   de forma **independiente** (se pueden entrenar en paralelo).
3. La predicción final se agrega:
   - **Regresión**: promedio de las $B$ predicciones.
   - **Clasificación**: voto mayoritario (o promedio de probabilidades).

Bagging reduce principalmente la **varianza** del modelo, porque promediar
muchos modelos con errores no perfectamente correlacionados cancela parte
del ruido individual de cada uno.

### 2.2 Random Forest

Random Forest es bagging aplicado a árboles, con un ingrediente adicional de
aleatoriedad: **en cada split**, en lugar de buscar la variable "óptima"
entre *todas* las variables disponibles, el algoritmo busca la mejor entre
un **subconjunto aleatorio de $m$ variables** (típicamente $m \approx
\sqrt{p}$ para clasificación, con $p$ = número total de variables).

Esta doble aleatoriedad (bootstrap de filas + subconjunto aleatorio de
columnas en cada split) **decorrelaciona** los árboles del ensamble entre sí,
lo que reduce la varianza más de lo que lograría bagging por sí solo (si
todos los árboles tienden a usar la misma variable "fuerte" en la raíz,
estarían muy correlacionados y promediarlos ayudaría menos).

**Ventajas**: robusto a ruido y outliers, funciona bien en datasets grandes
y de alta dimensión, entrega una medida de *feature importance*, requiere
poco tuning para obtener un buen desempeño de partida.

**Desventajas**: pierde la interpretabilidad directa de un árbol individual
(ya no se puede "leer" el modelo como una serie de reglas simples); más
costoso computacionalmente que un solo árbol.

### 2.3 Boosting / Gradient Boosting

A diferencia de bagging (árboles entrenados **en paralelo**, de forma
independiente), en **boosting** los árboles se entrenan **secuencialmente**:
cada nuevo árbol se enfoca en corregir los errores (residuales) del ensamble
construido hasta el momento.

En **Gradient Boosting** específicamente, esto se formaliza como un
**descenso de gradiente en el espacio de funciones**: en cada iteración se
ajusta un nuevo árbol a (una aproximación de) el gradiente negativo de la
función de pérdida respecto a las predicciones actuales del ensamble, y se
suma al ensamble con un factor de aprendizaje (`learning_rate`) pequeño.

Boosting tiende a reducir tanto **sesgo** como **varianza**, pero es más
sensible a overfitting si se usan demasiadas iteraciones o un
`learning_rate` demasiado alto, y al ser secuencial es más difícil de
paralelizar que bagging.

> **Nota — XGBoost / LightGBM / CatBoost:** estas son implementaciones
> optimizadas de gradient boosting (histogram binning, manejo nativo de
> categóricas, regularización adicional, paralelización a nivel de split,
> soporte de GPU, etc.) que se han convertido en **estado del arte en la
> industria** para datos tabulares, ganando sistemáticamente competencias de
> Kaggle y usándose en producción en muchas empresas. En este curso **no las
> instalamos** (requieren dependencias de sistema como `libomp` que no están
> garantizadas en todas las máquinas); en su lugar usamos
> `HistGradientBoostingClassifier` de scikit-learn, que implementa la misma
> idea central (gradient boosting con *histogram binning*, inspirado
> directamente en LightGBM) sin dependencias externas.

### 2.4 Demo: Random Forest y Gradient Boosting sobre el dataset de fraude

In [ ]:
rf_fraude = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf_fraude.fit(X_train_fraude, y_train_fraude)
y_pred_rf = rf_fraude.predict(X_test_fraude)

hgb = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
hgb.fit(X_train_fraude, y_train_fraude)
y_pred_hgb = hgb.predict(X_test_fraude)

print("== Random Forest ==")
print(classification_report(y_test_fraude, y_pred_rf, digits=3, target_names=["normal", "fraude"]))
print("== Gradient Boosting (HistGradientBoostingClassifier) ==")
print(classification_report(y_test_fraude, y_pred_hgb, digits=3, target_names=["normal", "fraude"]))

In [ ]:
def resumen_metricas(nombre, y_true, y_pred, y_score=None):
    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, y_score) if y_score is not None else np.nan,
    }

comparacion = pd.DataFrame([
    resumen_metricas("Árbol de decisión", y_test_fraude, y_pred_arbol, arbol.predict_proba(X_test_fraude)[:, 1]),
    resumen_metricas("Random Forest", y_test_fraude, y_pred_rf, rf_fraude.predict_proba(X_test_fraude)[:, 1]),
    resumen_metricas("Gradient Boosting", y_test_fraude, y_pred_hgb, hgb.predict_proba(X_test_fraude)[:, 1]),
]).set_index("modelo")

comparacion.round(3)

Los ensambles (Random Forest y Gradient Boosting) típicamente superan al
árbol individual en recall/F1/PR-AUC, a costa de perder la interpretabilidad
directa del árbol simple.

---

## 7. Ajuste de hiperparámetros

Hasta ahora usamos los hiperparámetros por defecto (o elegidos "a ojo") de
Random Forest y Gradient Boosting. En la práctica se busca la combinación de
hiperparámetros que optimiza una métrica de validación cruzada. Tres
herramientas para eso:

- **`GridSearchCV`**: prueba **todas** las combinaciones de una grilla de
  hiperparámetros. Exhaustivo pero costoso si el espacio de búsqueda es
  grande (crece exponencialmente con el número de hiperparámetros).
- **`RandomizedSearchCV`**: en lugar de probar todas las combinaciones,
  muestrea aleatoriamente un número fijo (`n_iter`) de combinaciones desde
  las distribuciones especificadas. Casi siempre encuentra una combinación
  casi tan buena como grid search completo, con mucho menos cómputo,
  especialmente cuando algunos hiperparámetros importan poco.
- **Optuna**: usa un enfoque *define-by-run* (el espacio de búsqueda se
  define dinámicamente en código Python, permitiendo condicionales entre
  hiperparámetros) y **optimización bayesiana** — por defecto, el algoritmo
  **TPE** (Tree-structured Parzen Estimator): en vez de muestrear
  combinaciones al azar, modela probabilísticamente qué regiones del
  espacio de hiperparámetros tienden a dar mejores resultados, y concentra
  ahí las siguientes pruebas. Con pocos *trials* suele acercarse más al
  óptimo que una búsqueda aleatoria pura.

### 3.1 Demo: `RandomizedSearchCV` sobre Random Forest

In [ ]:
from scipy.stats import randint

param_distributions = {
    "n_estimators": randint(100, 400),
    "max_depth": [None, 4, 8, 12, 16],
    "min_samples_leaf": randint(1, 20),
    "max_features": ["sqrt", "log2", None],
}

busqueda = RandomizedSearchCV(
    # n_jobs=-1 va en la búsqueda (paraleliza combinaciones de hiperparámetros);
    # dejamos el estimador base en n_jobs=1 para evitar paralelismo anidado.
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_distributions=param_distributions,
    n_iter=15,
    scoring="average_precision",   # PR-AUC: apropiada para clases desbalanceadas
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
busqueda.fit(X_train_fraude, y_train_fraude)

print("Mejores hiperparámetros encontrados:")
print(busqueda.best_params_)
print(f"Mejor PR-AUC en validación cruzada: {busqueda.best_score_:.3f}")

In [ ]:
rf_tuned = busqueda.best_estimator_
y_pred_rf_tuned = rf_tuned.predict(X_test_fraude)

comparacion.loc["Random Forest (RandomizedSearchCV)"] = resumen_metricas(
    "Random Forest (RandomizedSearchCV)",
    y_test_fraude, y_pred_rf_tuned, rf_tuned.predict_proba(X_test_fraude)[:, 1],
)
comparacion.round(3)

### 3.2 Demo: Optuna sobre Random Forest

Mismo espacio de búsqueda que arriba, pero con Optuna: en vez de muestrear
al azar, cada nuevo *trial* usa lo aprendido de los anteriores para
proponer combinaciones más prometedoras.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # silencia el log por trial


def objetivo(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_categorical("max_depth", [None, 4, 8, 12, 16]),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
    }
    modelo = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **params)
    return cross_val_score(
        modelo, X_train_fraude, y_train_fraude, cv=3, scoring="average_precision"
    ).mean()


estudio = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
estudio.optimize(objetivo, n_trials=15)

print("Mejores hiperparámetros encontrados (Optuna):")
print(estudio.best_params)
print(f"Mejor PR-AUC en validación cruzada: {estudio.best_value:.3f}")

In [ ]:
rf_optuna = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **estudio.best_params)
rf_optuna.fit(X_train_fraude, y_train_fraude)
y_pred_rf_optuna = rf_optuna.predict(X_test_fraude)

comparacion.loc["Random Forest (Optuna)"] = resumen_metricas(
    "Random Forest (Optuna)",
    y_test_fraude, y_pred_rf_optuna, rf_optuna.predict_proba(X_test_fraude)[:, 1],
)
comparacion.round(3)

---

## Resumen de conceptos clave

- El **sesgo** es el error por simplificar de más; la **varianza** es la
  sensibilidad del modelo a la muestra de entrenamiento particular. A mayor
  flexibilidad del modelo, generalmente menor sesgo pero mayor varianza.
- **Underfitting**: el modelo falla incluso en entrenamiento (modelo muy
  simple). **Overfitting**: el modelo memoriza el entrenamiento pero no
  generaliza (modelo muy complejo, pocos datos, mucho ruido).
- Existe un **trade-off entre interpretabilidad y precisión**: modelos
  simples se explican fácil pero pueden tener más sesgo; modelos complejos
  suelen ser más precisos pero más difíciles de auditar.
- El error de entrenamiento **subestima** el error real. La **validación
  cruzada** (K-Fold, típicamente k=5 o 10; o LOOCV, el caso extremo k=n) da
  una estimación más confiable y estable del desempeño de un modelo sobre
  datos nuevos.
- **Árboles de decisión**: particionan el espacio de entrada usando Gini o
  Entropía (clasificación) / varianza (regresión) como función de costo;
  son interpretables y no requieren normalización, pero necesitan
  regularización (pre-pruning o post-pruning) para no sobreajustar.
- **Métodos de ensamble**: bagging (Random Forest) entrena árboles en
  paralelo sobre muestras bootstrap + subconjuntos aleatorios de variables
  para reducir varianza; boosting (Gradient Boosting / XGBoost / LightGBM /
  CatBoost) entrena árboles secuencialmente, cada uno corrigiendo los
  errores del ensamble anterior.
- **Ajuste de hiperparámetros**: `GridSearchCV` (exhaustivo) y
  `RandomizedSearchCV` (muestreo aleatorio, más eficiente); herramientas
  como Optuna llevan esta idea más lejos con optimización bayesiana (TPE).

### Próxima sesión

En la Sesión 3 veremos **desbalance de clases** (sobre este mismo dataset
de fraude) y **series de tiempo** — dos temas distintos que comparten una
misma lección: la validación estándar (accuracy, K-Fold aleatorio) falla
en ambos casos, y hace falta una alternativa específica para cada uno.

---

# Ejercicio

Las partes 1-2 se resuelven sobre el dataset de fraude submuestreado ya
cargado arriba (`X_train_fraude`, `X_test_fraude`, `y_train_fraude`,
`y_test_fraude`, `rf_fraude`, `hgb`). La parte 3 se resuelve sobre `logreg`
y los datos de crédito (`X`, `y`) del inicio del notebook.

| # | Parte | Tiempo sugerido |
|---|---|---|
| 1 | Profundidad del árbol y overfitting | 10 min |
| 2 | Curvas Precision-Recall superpuestas: Random Forest vs. Gradient Boosting | 10 min |
| 3 | Comparar distintos valores de k en K-Fold | 10 min |

Si una parte se atasca, pasen a la siguiente: valen más las tres intentadas que una perfecta.

### Ejercicio 1 — Profundidad del árbol y overfitting

Entrena tres `DecisionTreeClassifier` con `max_depth` en `{2, 6, None}`
(`None` = sin límite de profundidad). Para cada uno, calcula F1 **en train**
y F1 **en test**. Grafica (o imprime en una tabla) cómo cambia la brecha
train/test a medida que el árbol crece: ¿en qué punto empieza a verse
overfitting (F1 en train mucho más alto que F1 en test)?

In [ ]:
# TODO: entrena DecisionTreeClassifier con max_depth en [2, 6, None]
# TODO: calcula f1_score en train y test para cada uno
# TODO: arma una tabla/gráfico comparando la brecha train-test

### Ejercicio 2 — Curvas Precision-Recall superpuestas: Random Forest vs. Gradient Boosting

Usando los modelos `rf_fraude` y `hgb` ya entrenados arriba, calcula la curva
Precision-Recall de cada uno sobre el test set (`precision_recall_curve`) y
grafícalas **superpuestas** en el mismo eje, incluyendo el PR-AUC de cada
modelo en la leyenda. ¿Cuál domina al otro en la mayoría de los niveles de
recall?

In [ ]:
# TODO: calcula precision_recall_curve para rf y hgb sobre el test set
# TODO: grafica ambas curvas superpuestas con matplotlib, con el PR-AUC en la leyenda

### Ejercicio 3 — Comparar distintos valores de k en K-Fold

Repite la validación cruzada K-Fold sobre `logreg` con distintos valores de
`k` (por ejemplo, 3, 5 y 10), usando `scoring="f1"`. Para cada `k`, calcula la
media y la desviación estándar del F1 entre folds. ¿Qué le pasa a la
desviación estándar a medida que `k` aumenta? ¿Por qué crees que ocurre eso?

In [ ]:
# TODO: 1) para k en [3, 5, 10], crea un KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
# TODO: 2) calcula cross_val_score(logreg, X, y, cv=kf, scoring="f1") para cada k
# TODO: 3) reporta media y desviación estándar del F1 para cada k en una tabla
# TODO: 4) concluye qué pasa con la varianza de la estimación a medida que k crece

---

### Ejercicio extra (opcional, sin cronometrar) — Ajustar el umbral de decisión

Por defecto, un clasificador predice la clase positiva cuando la probabilidad
estimada supera 0.5. Usando las probabilidades de `logreg` sobre `X_test`
(`logreg.predict_proba(X_test)[:, 1]`), calcula precisión y recall (de la
clase `bad`) para al menos 3 umbrales distintos (por ejemplo 0.3, 0.5 y 0.7).
¿Qué umbral usarías si el banco quiere **minimizar el riesgo** de aprobar
créditos malos, es decir, maximizar el recall de `bad`, aun a costa de
precisión?

In [ ]:
# TODO: 1) obtén las probabilidades predichas por logreg sobre X_test para la clase 'bad'
# TODO: 2) para umbrales = [0.2, 0.3, 0.5, 0.7, 0.8], calcula y_pred = (proba >= umbral)
# TODO: 3) calcula precision_score y recall_score para cada umbral y arma una tabla
# TODO: 4) concluye qué umbral recomendarías si el banco prioriza recall sobre precisión